# Scalable Analysis and Demand Prediction of NYC Taxi Trip Data Using Apache Spark

**CS-GY 6513 Big Data — Spring 2026**  
**Team:** Ninad Rade (nr3263), Roshi Bhati (rb6161), Prachiti Kulkarni (pk3117)

This notebook is the end-to-end walkthrough of the project pipeline:

1. Spark session setup
2. Data ingestion (Jan–Mar 2023 Yellow Taxi parquet files)
3. Preprocessing & feature engineering
4. Descriptive analytics (Spark SQL / DataFrame API)
5. Predictive modeling with Spark MLlib (demand regression)
6. Anomaly detection (rule-based + IQR)
7. Visualization & interpretation

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')

from src.spark_session import get_spark
from src.config import FIG_DIR, RESULTS_DIR, DATA_DIR

spark = get_spark()
print('Spark version :', spark.version)
print('Master        :', spark.sparkContext.master)
print('Data dir      :', DATA_DIR)

## 2. Data Ingestion

Load three monthly parquet files (Jan/Feb/Mar 2023). Spark reads the whole directory lazily and pushes down the parquet schema.

In [ ]:
from src.ingestion import load_trips, load_zones, attach_zone_names

trips_raw = load_trips(spark)
zones = load_zones(spark)
trips_raw = attach_zone_names(trips_raw, zones)
trips_raw.printSchema()

In [ ]:
trips_raw.select('tpep_pickup_datetime','tpep_dropoff_datetime','passenger_count',
                 'trip_distance','fare_amount','total_amount','pickup_zone','pickup_borough'
                ).show(5, truncate=False)

## 3. Preprocessing & Feature Engineering

Drop nulls, remove invalid / out-of-range records, derive `trip_duration_min`, `pickup_hour`, `pickup_day_of_week`, `is_weekend`, and `avg_speed_mph`.

In [ ]:
from src.preprocessing import preprocess

clean = preprocess(trips_raw)
clean.select('tpep_pickup_datetime','trip_duration_min','pickup_hour',
             'pickup_day_of_week','is_weekend','avg_speed_mph').show(5)

## 4. Descriptive Analytics

Each helper returns a Pandas DataFrame and is written to CSV in `outputs/results/`.

In [ ]:
from src.descriptive import run_all as run_descriptive

desc = run_descriptive(clean, RESULTS_DIR)
desc['summary_statistics']

In [ ]:
desc['trips_per_hour']

In [ ]:
desc['busiest_pickup_zones']

### 4.1 Visualizations

In [ ]:
from src import visualization as viz

viz.plot_trips_per_hour(desc['trips_per_hour'], FIG_DIR)
viz.plot_trips_per_day(desc['trips_per_day'], FIG_DIR)
viz.plot_trips_per_dow(desc['trips_per_day_of_week'], FIG_DIR)
viz.plot_busiest_zones(desc['busiest_pickup_zones'], FIG_DIR, kind='pickup')
viz.plot_busiest_zones(
    desc['busiest_dropoff_zones'].rename(columns={'dropoff_zone':'pickup_zone','dropoff_borough':'pickup_borough'}),
    FIG_DIR, kind='dropoff'
)
viz.plot_passenger_distribution(desc['passenger_count_distribution'], FIG_DIR)
viz.plot_payment_breakdown(desc['payment_type_breakdown'], FIG_DIR)
viz.plot_hour_zone_heatmap(desc['hour_zone_heatmap'], FIG_DIR)

fare_sample = clean.select('trip_distance','fare_amount').sample(False, 0.002, seed=1).toPandas()
viz.plot_fare_vs_distance(fare_sample, FIG_DIR)
print('All descriptive plots saved to', FIG_DIR)

## 5. Spark SQL Queries

The same questions answered through SQL for clarity.

In [ ]:
clean.createOrReplaceTempView('trips')

spark.sql('''
  SELECT pickup_hour,
         COUNT(*) AS trips,
         ROUND(AVG(fare_amount), 2) AS avg_fare,
         ROUND(AVG(trip_distance), 2) AS avg_distance
  FROM trips
  GROUP BY pickup_hour
  ORDER BY trips DESC
  LIMIT 5
''').show()

In [ ]:
spark.sql('''
  SELECT pickup_borough, COUNT(*) AS trips,
         ROUND(AVG(fare_amount),2) AS avg_fare
  FROM trips
  WHERE pickup_borough IS NOT NULL
  GROUP BY pickup_borough
  ORDER BY trips DESC
''').show()

## 6. Predictive Modeling (Spark MLlib)

We aggregate trips to hourly demand per pickup zone and train Linear Regression and Random Forest regressors to predict the `demand` count.

In [ ]:
from src.predictive import train_demand_models, feature_importance_pdf, save_predictions_sample

model_out = train_demand_models(clean)
pd.DataFrame(model_out['metrics']).T

In [ ]:
viz.plot_model_metrics(model_out['metrics'], FIG_DIR)

feature_names = ['pickup_hour','pickup_day_of_week','is_weekend',
                 'PULocationID','borough_idx','avg_distance','avg_fare']
fi = feature_importance_pdf(model_out['rf_model'], feature_names)
fi.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)
viz.plot_feature_importance(fi, FIG_DIR)
fi

In [ ]:
preds = save_predictions_sample(
    model_out['rf_model'], model_out['test'],
    RESULTS_DIR / 'predictions_sample.csv'
)
viz.plot_predicted_vs_actual(preds, FIG_DIR)
preds.head()

## 7. Anomaly Detection

Combines rule-based thresholds (e.g., fare > $500, duration > 3h, zero-distance paid) and IQR outliers on `fare_amount`, `trip_distance`, `trip_duration_min`.

In [ ]:
from src.anomaly import detect as detect_anomalies

anom = detect_anomalies(clean, RESULTS_DIR)
anom['summary']

In [ ]:
anom['rule_counts']

In [ ]:
viz.plot_anomaly_rule_counts(anom['rule_counts'], FIG_DIR)
box_sample = clean.select('fare_amount','trip_distance','trip_duration_min').sample(False, 0.002, seed=2).toPandas()
viz.plot_fare_box_with_bounds(box_sample, anom['bounds'], FIG_DIR)

In [ ]:
anom['sample'].head(10)

## 8. Conclusions

- Spark processed ~10M trip records from Jan–Mar 2023 in a few minutes on a local 4-core machine.
- Demand concentrates sharply in evening rush (17:00–19:00) and on weekdays; weekends show later-night peaks.
- Midtown Center, Upper East Side, and JFK/LGA dominate pickup volume.
- The Random Forest regressor clearly beats Linear Regression on the demand forecasting task.
- ~1–2% of trips are flagged as anomalies — the dominant classes are extreme fares and zero-distance paid trips.

All CSV aggregates are saved under `outputs/results/` and all figures under `outputs/figures/`.

In [ ]:
spark.stop()